In [1]:
import os
import json
import torch
import argparse
import numpy as np
from dataset import EurDataset, collate_data
from models.transceiver import DeepSC
from torch.utils.data import DataLoader
from utils import BleuScore, SNR_to_noise, greedy_decode, SeqtoText, Channels,PowerNormalize,subsequent_mask
from tqdm import tqdm
from sklearn.preprocessing import normalize
# from bert4keras.backend import keras
# from bert4keras.models import build_bert_model
# from bert4keras.tokenizers import Tokenizer
from w3lib.html import remove_tags
from student import Student


In [2]:
parser = argparse.ArgumentParser()
parser.add_argument('--data-dir', default='europarl/train_data.pkl', type=str)
parser.add_argument('--vocab-file', default='europarl/vocab.json', type=str)
parser.add_argument('--checkpoint-path', default='checkpoints/deepsc-Rayleigh', type=str)
parser.add_argument('--channel', default='Rayleigh', type=str)
parser.add_argument('--MAX-LENGTH', default=30, type=int)
parser.add_argument('--MIN-LENGTH', default=4, type=int)
parser.add_argument('--d-model', default=128, type = int)
parser.add_argument('--dff', default=512, type=int)
parser.add_argument('--num-layers', default=4, type=int)
parser.add_argument('--num-heads', default=8, type=int)
parser.add_argument('--batch-size', default=64, type=int)
parser.add_argument('--epochs', default=2, type = int)
parser.add_argument('--bert-config-path', default='bert/cased_L-12_H-768_A-12/bert_config.json', type = str)
parser.add_argument('--bert-checkpoint-path', default='bert/cased_L-12_H-768_A-12/bert_model.ckpt', type = str)
parser.add_argument('--bert-dict-path', default='bert/cased_L-12_H-768_A-12/vocab.txt', type = str)

_StoreAction(option_strings=['--bert-dict-path'], dest='bert_dict_path', nargs=None, const=None, default='bert/cased_L-12_H-768_A-12/vocab.txt', type=<class 'str'>, choices=None, required=False, help=None, metavar=None)

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [4]:
test_eur = EurDataset('test')
print(test_eur[0])

[1, 21715, 18169, 13612, 1824, 19619, 15947, 13501, 2710, 13088, 4634, 21827, 21660, 21933, 21124, 6184, 838, 1270, 13088, 21332, 20114, 925, 9362, 15947, 9883, 22047, 4, 2]


In [5]:
def text_to_indices(text, token_to_idx, start_idx, end_idx, pad_idx, max_length):
    tokens = text.lower().strip().split()
    unk_idx = token_to_idx.get("<UNK>", None)

    seq = [start_idx]
    for tok in tokens:
        if tok in token_to_idx:
            seq.append(token_to_idx[tok])
        elif unk_idx is not None:
            seq.append(unk_idx)

    seq.append(end_idx)

    if len(seq) < max_length:
        seq += [pad_idx] * (max_length - len(seq))
    else:
        seq = seq[:max_length]
        seq[-1] = end_idx

    return seq




In [6]:
def transmitter(src, model,padding_idx, device):
    src_mask = (src == padding_idx).unsqueeze(-2).type(torch.FloatTensor).to(device) #[batch, 1, seq_len]

    enc_output = model.encoder(src, src_mask)
    channel_enc_output = model.channel_encoder(enc_output)
    Tx_sig = PowerNormalize(channel_enc_output)

    return Tx_sig

In [7]:
def receiver(Rx_sig, model, src, max_len, start_symbol, padding_idx, device):

    memory = model.channel_decoder(Rx_sig)
    
    outputs = torch.ones(src.size(0), 1).fill_(start_symbol).type_as(src.data)

    for i in range(max_len - 1):
        # create the decode mask
        trg_mask = (outputs == padding_idx).unsqueeze(-2).type(torch.FloatTensor) #[batch, 1, seq_len]
        look_ahead_mask = subsequent_mask(outputs.size(1)).type(torch.FloatTensor)
#        print(look_ahead_mask)
        combined_mask = torch.max(trg_mask, look_ahead_mask)
        combined_mask = combined_mask.to(device)

        # decode the received signal
        dec_output = model.decoder(outputs, memory, combined_mask, None)
        pred = model.dense(dec_output)
        
        # predict the word
        prob = pred[: ,-1:, :]  # (batch_size, 1, vocab_size)
        #prob = prob.squeeze()

        # return the max-prob index
        _, next_word = torch.max(prob, dim = -1)
        #next_word = next_word.unsqueeze(1)
        
        outputs = torch.cat([outputs, next_word], dim=1)

    return outputs




In [16]:
def student_decode(Rx_sig, model, src, max_len, start_symbol, padding_idx, device):
    memory = model.channel_decoder(Rx_sig)
    
    outputs = torch.ones(src.size(0), 1).fill_(start_symbol).type_as(src.data)

    for i in range(max_len - 1):
        # create the decode mask
        trg_mask = (outputs == padding_idx).unsqueeze(-2).type(torch.FloatTensor) #[batch, 1, seq_len]
        look_ahead_mask = subsequent_mask(outputs.size(1)).type(torch.FloatTensor)
#        print(look_ahead_mask)
        combined_mask = torch.max(trg_mask, look_ahead_mask)
        combined_mask = combined_mask.to(device)

        # decode the received signal
        dec_output = model.decoder(outputs, memory, combined_mask, None)
        pred = model.dense(dec_output)
        
        # predict the word
        prob = pred[: ,-1:, :]  # (batch_size, 1, vocab_size)
        #prob = prob.squeeze()

        # return the max-prob index
        _, next_word = torch.max(prob, dim = -1)
        #next_word = next_word.unsqueeze(1)
        
        outputs = torch.cat([outputs, next_word], dim=1)

    return outputs

In [19]:
def interactive_demo(args, net, student, text, token_to_idx, start_idx, end_idx, pad_idx, device, snr_in=6.0):
    print("DeepSC interactive mode")

    channels = Channels()
    StoT = SeqtoText(token_to_idx, end_idx)

    try:
        snr = float(snr_in)
    except ValueError:
        snr = 4.0
    
    noise_std = SNR_to_noise(snr)

    channel = args.channel
    
    seq = text_to_indices(
        text=text,
        token_to_idx=token_to_idx,
        start_idx=start_idx,
        end_idx=end_idx,
        pad_idx=pad_idx,
        max_length=args.MAX_LENGTH
    )

    src = torch.tensor([seq], dtype=torch.long).to(device)

    net.eval()
    student.eval()
    with torch.no_grad():
        Tx_sig = transmitter(
            src=src,
            model=net,
            padding_idx=pad_idx,
            device=device
        )

        print("Transmitting ...")
        print(Tx_sig.shape)

        if channel == 'AWGN':
            Rx_sig = channels.AWGN(Tx_sig, noise_std)
        elif channel == 'Rayleigh':
            Rx_sig = channels.Rayleigh(Tx_sig, noise_std)
        elif channel == 'Rician':
            Rx_sig = channels.Rician(Tx_sig, noise_std)
        else:
            raise ValueError("Please choose from AWGN, Rayleigh, and Rician")
                


        out = receiver(
            Rx_sig=Rx_sig,
            model=net,
            src=src,
            max_len=args.MAX_LENGTH,
            start_symbol=start_idx,
            padding_idx=pad_idx,
            device=device
        )

        student_out = student_decode(
            Rx_sig=Rx_sig,
            model=student,
            src=src,
            max_len=args.MAX_LENGTH,
            start_symbol=start_idx,
            padding_idx=pad_idx,
            device=device
        )
    print("Receiving ...")
    print(out.shape)
    print("-" * 50)

    

    decoded_ids = out.cpu().numpy().tolist()[0]
    student_decoded_ids = student_out.cpu().numpy().tolist()[0]
    received = StoT.sequence_to_text(decoded_ids)
    student_received = StoT.sequence_to_text(student_decoded_ids)
    print("Original :", text)
    print("Received :", received)
    print("Student Received :", student_received)
    print("-" * 50)

In [ ]:
from types import SimpleNamespace

if __name__ == '__main__':
    
    my_vars = {
        'checkpoint_path': 'checkpoints/deepsc-Rayleigh',
        'data_dir': 'data/train/europarl/train_data.pkl',
        'vocab_file': 'data/train/europarl/vocab.json',
        'channel': 'Rayleigh',
        'MAX_LENGTH': 30,
        'MIN_LENGTH': 4,
        'd_model': 128,
        'dff': 512,
        'num_layers': 4,
        'num_heads': 8,
        'batch_size': 64,
        'epochs': 2
    }

    my_vars = SimpleNamespace(**my_vars)

    vocab = json.load(open(my_vars.vocab_file, 'rb'))

    token_to_idx = vocab['token_to_idx']
    idx_to_token = dict(zip(token_to_idx.values(), token_to_idx.keys()))
    num_vocab = len(token_to_idx)

    pad_idx = token_to_idx["<PAD>"]
    start_idx = token_to_idx["<START>"]
    end_idx = token_to_idx["<END>"]

    deepsc = DeepSC(
        my_vars.num_layers,
        num_vocab,
        num_vocab,
        num_vocab,
        num_vocab,
        my_vars.d_model,
        my_vars.num_heads,
        my_vars.dff,
        0.1
    ).to(device)

    student =  Student(
        2,
        num_vocab,
        num_vocab,
        num_vocab,
        num_vocab,
        my_vars.d_model,
        my_vars.num_heads,
        my_vars.dff,
        0.1
    ).to(device)

    model_path = 'checkpoints/deepsc-Rayleigh/checkpoint_100.pth'
    student_path = 'checkpoints/tr_kd/student_tr_best.pth'

    checkpoint = torch.load(model_path, map_location=device)
    # print(checkpoint.keys())
    deepsc.load_state_dict(checkpoint)

    student_checkpoint = torch.load(student_path, map_location=device)
    # print(student_checkpoint.keys())
    student.channel_decoder.load_state_dict(student_checkpoint['channel_decoder'])
    student.decoder.load_state_dict(student_checkpoint['decoder'])
    student.dense.load_state_dict(student_checkpoint['dense'])

    deepsc.eval()
    student.eval()

    deepsc.encoder.eval()
    deepsc.decoder.eval()

    print("model loaded")

    text = "The European Union is a political and economic union."
    text2 = "I declare resumed the session adjourned on 17 December 1999. Belated happy new year"
    text3 = "the is a happy new year."

    text4 = "they accept agreement agreement agreement and adapt system and improve performance"
    text5="I declare resumed the session of the European Parliament adjourned on Friday 17 December 1999"

    interactive_demo(my_vars, deepsc, student, text6, token_to_idx, start_idx, end_idx, pad_idx, device)

odict_keys(['encoder.embedding.weight', 'encoder.pos_encoding.pe', 'encoder.enc_layers.0.mha.wq.weight', 'encoder.enc_layers.0.mha.wq.bias', 'encoder.enc_layers.0.mha.wk.weight', 'encoder.enc_layers.0.mha.wk.bias', 'encoder.enc_layers.0.mha.wv.weight', 'encoder.enc_layers.0.mha.wv.bias', 'encoder.enc_layers.0.mha.dense.weight', 'encoder.enc_layers.0.mha.dense.bias', 'encoder.enc_layers.0.ffn.w_1.weight', 'encoder.enc_layers.0.ffn.w_1.bias', 'encoder.enc_layers.0.ffn.w_2.weight', 'encoder.enc_layers.0.ffn.w_2.bias', 'encoder.enc_layers.0.layernorm1.weight', 'encoder.enc_layers.0.layernorm1.bias', 'encoder.enc_layers.0.layernorm2.weight', 'encoder.enc_layers.0.layernorm2.bias', 'encoder.enc_layers.1.mha.wq.weight', 'encoder.enc_layers.1.mha.wq.bias', 'encoder.enc_layers.1.mha.wk.weight', 'encoder.enc_layers.1.mha.wk.bias', 'encoder.enc_layers.1.mha.wv.weight', 'encoder.enc_layers.1.mha.wv.bias', 'encoder.enc_layers.1.mha.dense.weight', 'encoder.enc_layers.1.mha.dense.bias', 'encoder.enc_